# 01 — EDA & Data Preparation
## Hospital Facility Location — Canton of Vaud

**Master in Sustainability Management and Technologies — Logistics project**

## 1. Introduction

The Canton of Vaud has **50 hospitals / facilities** and **300 communes**. Our project asks: which subset of hospitals should remain open to serve the population, when we take into account

- the **running cost** of each hospital,
- the **bed capacity** of each hospital,
- the **distance** between communes and hospitals,
- the **operational emissions** of hospitals (Scope 1 + 2 type),
- the **transport emissions** generated by patients travelling to a hospital,
- the **growth of demand** as the population evolves.

### What this notebook does

This first notebook is the **EDA & data-preparation step**. It loads the raw files, fixes encoding and name-matching issues, builds the clean datasets, and produces the descriptive statistics and plots that the final report needs.

The optimization models themselves are in the second notebook:
`02_Optimization_Hospital_Facility_Location_Vaud.ipynb`.

### Three important methodological decisions (validated)

1. **Capacity** — we use `Beds_Total` from `costs_hospitals_full.csv` as the per-facility bed count. This is the true per-facility capacity (CHUV = 1175 beds, Psychiatrie Nord CPNVD = 50 beds, etc.). The previous approach of equally splitting institution-aggregated beds across facilities is not used because it creates unrealistic identical capacities for facilities of very different sizes.

2. **Demand** — the bed demand per commune is computed as
$$ \text{Demand}_i = \text{Population}_i \times 0.164 \times \frac{5}{365} $$
This represents the **average number of beds simultaneously occupied** by the population of commune $i$, assuming that 16.4 % of the population requires a hospital stay of 5 days per year on average. This gives ≈ 2.25 beds per 1000 inhabitants, in line with current Swiss acute-care figures.

3. **Cost-minimization objective** — for every cost model (notebook 2), we minimize
$$ \min \sum_j C_j \, y_j $$
where $C_j$ is the running cost of hospital $j$. We do **not** minimize the number of hospitals: the count is reported as a KPI only.

### Optimization granularity

All optimization is done at the **commune × hospital** level (300 × 50). Districts are used **only** for reporting, aggregation, and interpretation — never as decision units.

### File naming note

The cost file is originally called `costs hospitals_full.csv` (with a space). Spaces are problematic in URLs and Colab paths. We assume the file has been renamed to `costs_hospitals_full.csv` in the GitHub repo. If you must keep the original name, replace the URL fragment with `"costs%20hospitals_full.csv"`.


## 2. Load libraries

In [ ]:
# Install OR-Tools now (used in notebook 2; harmless to install here too)
!pip install ortools -q


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 3. Key parameters

All the assumptions are gathered in one place so they can be changed easily.


In [ ]:
# --- Demand assumption ---
# 16.4% of the population requires a hospital stay of 5 days per year
# => Demand_i = Population_i * 0.164 * 5 / 365  (average beds simultaneously occupied)
HOSPITALIZATION_RATE  = 0.164    # share of population needing a stay per year (assumption)
AVG_LENGTH_OF_STAY    = 5        # days (assumption)
DAYS_PER_YEAR         = 365

# Equivalent figure expressed in beds per 1000 inhabitants
BEDS_PER_1000_EQUIV   = HOSPITALIZATION_RATE * AVG_LENGTH_OF_STAY / DAYS_PER_YEAR * 1000
print(f"Equivalent demand = {BEDS_PER_1000_EQUIV:.2f} beds per 1000 inhabitants")

# --- Transport emission factor (kg CO2 / passenger-km) ---
# 0.12 = typical average for a private car (EEA / mobitool). ASSUMPTION.
EMISSION_FACTOR_KM    = 0.12

# --- Default distance threshold for the baseline optimization (km) ---
D_DEFAULT_KM          = 16

# --- GitHub raw URL of the project data ---
BASE_URL = "https://raw.githubusercontent.com/Trickwillfrit/Sustainable-logistics/main/"


## 4. Load the raw data from GitHub

We load the five files. Encoding matters here:

- `Distance_matrix.csv` and `hospitals_full.csv` are in **Latin-1**.
- `costs_hospitals_full.csv` and `Hospital_emissions.csv` are in **UTF-8** and (the cost file) uses `;` as separator. If we read them as Latin-1, accented names appear as mojibake (`HÃ´pital`) and every later merge fails.


In [ ]:
# 1) Distance matrix (latin-1)
dist_raw = pd.read_csv(BASE_URL + "Distance_matrix.csv", encoding="latin1")

# 2) Hospital list (latin-1)
hosp_raw = pd.read_csv(BASE_URL + "hospitals_full.csv", encoding="latin1")

# 3) Cost file (UTF-8, ';' separator)
costs_raw = pd.read_csv(BASE_URL + "costs_hospitals_full.csv", sep=";", encoding="utf-8")

# 4) Emissions (UTF-8)
emis_raw = pd.read_csv(BASE_URL + "Hospital_emissions.csv", encoding="utf-8")

# 5) Population by commune (Excel, sheet "Total")
pop_raw = pd.read_excel(BASE_URL + "Pop_district.xlsx", sheet_name="Total")

print("Distance matrix :", dist_raw.shape)
print("Hospitals       :", hosp_raw.shape)
print("Cost file       :", costs_raw.shape)
print("Emissions       :", emis_raw.shape)
print("Population      :", pop_raw.shape)


## 5. Clean the hospital list (`hospitals_full.csv`)

We filter Canton VD and keep the location info and the **bed-type breakdown** (acute, psychiatric, rehab, birth) only for descriptive EDA. The **capacity used in the optimization** comes from the cost file (next section), not from here.

> Note. The bed counts in `hospitals_full.csv` are **institution-aggregated**: every facility of the same institution shows identical totals (e.g. CHUV institution shows 1300 beds on all 3 of its facilities). For descriptive plots this is fine if we are aware of it; for the capacity constraint of the optimization we must use the true per-facility values (`Beds_Total` from the cost file).


In [ ]:
# Keep only Vaud
hospitals = hosp_raw[hosp_raw["Canton__Facility_"] == "VD"].copy()

# Keep useful columns (we keep Institution to be able to deduplicate emissions later)
keep = ["Facility", "Town__Facility_", "Institution", "Canton__Facility_",
        "Beds_Acute_Care", "Beds_in_Psychiatric_Wards",
        "Beds_in_Rehab", "Beds_in_Birth_Centers",
        "longitude", "latitude"]
hospitals = hospitals[keep].reset_index(drop=True)

# Missing bed values -> 0 (the hospital simply does not offer that bed type)
bed_cols = ["Beds_Acute_Care", "Beds_in_Psychiatric_Wards",
            "Beds_in_Rehab", "Beds_in_Birth_Centers"]
hospitals[bed_cols] = hospitals[bed_cols].fillna(0)

# Bed-type total (descriptive only - INSTITUTION-aggregated, not per-facility)
hospitals["Beds_TypeTotal_InstAggr"] = hospitals[bed_cols].sum(axis=1)

print("Number of VD hospitals :", len(hospitals))
hospitals.head()


## 6. Clean the cost data (`costs_hospitals_full.csv`)

### What is `Cost_per_Bed` really?

Despite its name, `Cost_per_Bed` is **not** a unit cost — it is the **total annual running-cost proxy** for the facility. Three quick observations confirm this:

- CHUV: 1175 beds, value = 5,170,000 → ratio ≈ 4,400 / bed
- Hôpital de Morges: 215 beds, value = 838,500 → ratio ≈ 3,900 / bed
- Hôpital de Rennaz: 254 beds, value = 990,600 → ratio ≈ 3,900 / bed

The values scale with the number of beds, the ratio is constant inside each `Size_Category` (Small ≈ 2,100 / bed, Moderate ≈ 3,900, Big ≈ 3,900, Major Center ≈ 4,400), and a "unit cost" of 5,170,000 CHF per CHUV bed would be physically impossible. So:

$$ \text{Hospital\_Cost}_j = \text{Cost\_per\_Bed}_j $$

(we use the column **as-is** — we do **not** multiply by beds again, which would square the cost).

### Hospital name fixes

Two true typos prevent merging the cost & emissions files with the distance matrix:

| In cost / emissions file | In distance matrix (reference) |
|---|---|
| `Hôptial d'Aubonne` | `Hôpital d'Aubonne` |
| `Pôle santé du Pays-d_x0019_ Enhaut` | `Pôle Santé Pays-d'Enhaut` |

We fix them with a small `NAME_FIXES` dictionary applied to both the cost and emissions files.


In [ ]:
# Two known name typos
NAME_FIXES = {
    "Hôptial d'Aubonne": "Hôpital d'Aubonne",
    "Pôle santé du Pays-d_x0019_ Enhaut": "Pôle Santé Pays-d'Enhaut",
}

# Filter VD and keep useful columns
costs = costs_raw[costs_raw["Canton__Facility_"] == "VD"][
    ["Facility", "Beds_Aggregated", "Beds_Total", "Size_Category", "Cost_per_Bed"]
].copy()
costs["Facility"] = costs["Facility"].replace(NAME_FIXES)

# Hospital_Cost = Cost_per_Bed (already total)
costs["Hospital_Cost"] = costs["Cost_per_Bed"].astype(float)

# Beds_Total = true per-facility capacity (what we use in the optimization)
costs["Beds_Total"] = costs["Beds_Total"].fillna(0).astype(float)

print(f"VD hospitals in cost file: {len(costs)}")
print(f"\nSum of Beds_Total (per facility, used in optimization): {costs['Beds_Total'].sum():.0f}")
print(f"\nHospital_Cost statistics (CHF / year, total per hospital):")
print(costs["Hospital_Cost"].describe().apply(lambda x: f"{x:,.0f}"))
print(f"\nSize category counts:")
print(costs["Size_Category"].value_counts())
costs.head()


## 7. Clean the emissions data (`Hospital_emissions.csv`)

### How are emissions reported?

The `Emissions` column is **constant across all facilities of the same institution** (every CHUV facility shows 8143.47 t CO₂; every eHnv facility shows 561.71 t CO₂, etc.). So the data is reported at the **institution level**, repeated on each facility row.

If we naively sum the column we would count each institution multiple times — about 44 775 t CO₂ instead of the correct institutional total of about 18 852 t CO₂.

### Per-facility allocation (clearly stated assumption)

To use these numbers inside an LP/MIP at the facility level, we **allocate the institutional emissions across facilities proportionally to `Beds_Total`**:

$$ E_j = \frac{\text{Beds\_Total}_j}{\sum_{k \in \text{Inst}(j)} \text{Beds\_Total}_k} \times \text{Institution\_Emissions} $$

So if a facility holds 20% of the beds of its institution, it carries 20% of the institutional emissions. Facilities with 0 beds (administrative sites) get 0 emissions. This allocation:

- preserves the institutional total when all facilities of an institution are open,
- gives every "real" facility a positive emission cost in the optimization,
- avoids the need for institution-level binary variables (which would complicate the MIP).

**This is an assumption** — alternatives like equal split or employee-based allocation would also be defensible. The rule is documented here and can be changed for a sensitivity analysis in the final report.


In [ ]:
# Apply the same name fixes
emis = emis_raw.copy()
emis["Facility"] = emis["Facility"].replace(NAME_FIXES)

print(f"Rows in emissions file        : {len(emis)}")
print(f"Unique institutions           : {emis['Institution'].nunique()}")
print(f"Sum of column (DOUBLE-COUNTS) : {emis['Emissions'].sum():,.0f} t CO2")
print(f"Sum deduplicated by inst.     : {emis.drop_duplicates(subset=['Institution'])['Emissions'].sum():,.0f} t CO2")


In [ ]:
# --- Allocate institutional emissions proportionally to Beds_Total ---
# 1) attach per-facility beds from the cost file
em_b = emis.merge(costs[["Facility", "Beds_Total"]], on="Facility", how="left")

# 2) total beds per institution
inst_total_beds = em_b.groupby("Institution")["Beds_Total"].transform("sum")

# 3) fallback to equal split if institution has 0 total beds
n_fac_per_inst = em_b.groupby("Institution")["Facility"].transform("count")
ratio = np.where(inst_total_beds > 0,
                 em_b["Beds_Total"] / inst_total_beds,
                 1.0 / n_fac_per_inst)

em_b["Hospital_Emissions"] = em_b["Emissions"] * ratio

emissions = em_b[["Facility", "Institution", "Hospital_Emissions"]].copy()

# Sanity check
check = (emissions.merge(emis.drop_duplicates(subset=["Institution"])[["Institution","Emissions"]],
                         on="Institution", how="left")
                  .groupby("Institution")
                  .agg(sum_alloc=("Hospital_Emissions","sum"),
                       inst_emis=("Emissions","first")))
check["diff"] = (check["sum_alloc"] - check["inst_emis"]).round(3)
print("Sanity check: difference between sum of allocated emissions and institutional total:")
print(check["diff"].describe())
print(f"\nTotal allocated emissions = {emissions['Hospital_Emissions'].sum():,.0f} t CO2 (matches institutional total)")
emissions.head()


## 8. Clean the population data (`Pop_district.xlsx`)

### Year columns

The Excel sheet "Total" has 9 population columns named `Population au 31.12.`, `Population au 31.12..1`, …, `Population au 31.12..8`. The headers do not contain year numbers, so we **assume** the columns correspond to years **2016 → 2024** (oldest → newest). This is the typical ordering used by Statistique Vaud. The most recent column (`.8`) gives a canton-wide total of about 864 k, consistent with the official 2024 figure.

> This year mapping is an **assumption**. If the source publication clarifies it differently, just change the dictionary below — the rest of the code is unaffected.

### Cleaning steps

- rename columns,
- drop the grand-total row (`District == "Total"`) and the per-district subtotal rows (`Commune == "Total"`) — otherwise the canton population looks doubled (~1.7 M),
- strip whitespace,
- keep the historical population columns (we will use them for the demand-growth scenarios in notebook 2).


In [ ]:
# Year mapping (ASSUMPTION: 2016 ... 2024)
YEAR_COLS = {
    "Population au 31.12.":   "Pop_2016",
    "Population au 31.12..1": "Pop_2017",
    "Population au 31.12..2": "Pop_2018",
    "Population au 31.12..3": "Pop_2019",
    "Population au 31.12..4": "Pop_2020",
    "Population au 31.12..5": "Pop_2021",
    "Population au 31.12..6": "Pop_2022",
    "Population au 31.12..7": "Pop_2023",
    "Population au 31.12..8": "Pop_2024",   # latest = "current"
}

pop = pop_raw.rename(columns={"District ":"District", "Mesures":"Commune", **YEAR_COLS}).copy()
pop = pop[["District","Commune"] + list(YEAR_COLS.values())]

# Drop grand-total and per-district subtotals
pop = pop[pop["District"] != "Total"]
pop = pop[pop["Commune"]  != "Total"]
pop = pop.dropna(subset=["Commune"])

# Strip whitespace
pop["Commune"]  = pop["Commune"].astype(str).str.strip()
pop["District"] = pop["District"].astype(str).str.strip()

# All year columns to int
for c in YEAR_COLS.values():
    pop[c] = pd.to_numeric(pop[c], errors="coerce").fillna(0).astype(int)

# "Population" = most recent year (Pop_2024)
pop["Population"] = pop["Pop_2024"]

print(f"Number of communes        : {len(pop)}")
print(f"Total population 2024     : {pop['Population'].sum():,}")
print(f"Total population 2016     : {pop['Pop_2016'].sum():,}")
print(f"Overall growth 2016->2024 : {(pop['Pop_2024'].sum()/pop['Pop_2016'].sum()-1)*100:.1f} %")
pop.head()


## 9. Clean the distance matrix (`Distance_matrix.csv`)

We rename `ID` to `Commune`, strip whitespace (the source has trailing spaces like `"Renens "`), convert distances from **metres to kilometres**, and merge with the cleaned population table.

We also compute the bed demand per commune:
$$ \text{Demand}_i = \text{Population}_i \times 0.164 \times \frac{5}{365} $$


In [ ]:
dist = dist_raw.rename(columns={"ID": "Commune"}).copy()
dist["Commune"] = dist["Commune"].astype(str).str.strip()

# Hospital columns = everything except 'I' and 'Commune'
hospital_cols_dm = [c for c in dist.columns if c not in ["I", "Commune"]]

# Convert m -> km
dist[hospital_cols_dm] = dist[hospital_cols_dm] / 1000.0

# Merge population & district
dist = dist.merge(pop[["Commune","District"] + list(YEAR_COLS.values()) + ["Population"]],
                  on="Commune", how="left")

n_before = len(dist)
dist = dist.dropna(subset=["Population"]).reset_index(drop=True)
dist["Population"] = dist["Population"].astype(int)
print(f"Communes before merge: {n_before}, after merge: {len(dist)}")

# Bed demand (simultaneous occupancy)
dist["Demand_Beds"] = dist["Population"] * HOSPITALIZATION_RATE * AVG_LENGTH_OF_STAY / DAYS_PER_YEAR

print(f"\nTotal demand : {dist['Demand_Beds'].sum():,.1f} beds (simultaneous)")
dist[["Commune","District","Population","Demand_Beds"] + hospital_cols_dm[:2]].head()


## 10. Hospital name consistency checks

After the UTF-8 encoding fix (for the cost and emissions files) and the 2 name corrections (`NAME_FIXES`), all 50 hospitals must appear in all four files.


In [ ]:
dm_set    = set(hospital_cols_dm)
hosp_set  = set(hospitals["Facility"])
costs_set = set(costs["Facility"])
emis_set  = set(emissions["Facility"])

print(f"Hospitals in distance matrix : {len(dm_set)}")
print(f"Hospitals in hospitals_full  : {len(hosp_set)}")
print(f"Hospitals in costs           : {len(costs_set)}")
print(f"Hospitals in emissions       : {len(emis_set)}")
print()
print("In distance matrix but NOT in costs :", sorted(dm_set - costs_set))
print("In costs but NOT in distance matrix :", sorted(costs_set - dm_set))
print("In distance matrix but NOT in emis  :", sorted(dm_set - emis_set))
print("In emis but NOT in distance matrix  :", sorted(emis_set - dm_set))

assert dm_set == hosp_set == costs_set == emis_set, "Hospital name mismatch remaining!"
print("\n=> All 50 hospital names match across all files. OK.")


## 11. Build the master hospital table

We combine everything into a single table whose row order follows the **column order of the distance matrix**, so we can index by position later in the LP without any merge.

We also flag the single **administrative site** (`Samaritain Site administratif`, 0 beds and 0 cost) — for the optimization it will be excluded from the candidate set; otherwise a cost-minimizing model would always pick it "for free" without it contributing any treatment capacity.


In [ ]:
master = (hospitals
          .merge(costs, on="Facility", how="left")
          .merge(emissions, on="Facility", how="left"))

# Reorder rows to follow the distance-matrix column order
master = master.set_index("Facility").loc[hospital_cols_dm].reset_index()

# Flag administrative / zero-capacity / zero-cost facilities
master["Is_Administrative"] = (master["Beds_Total"] == 0) | (master["Hospital_Cost"] == 0)
print("Administrative / zero-cost / zero-bed facilities:")
print(master[master["Is_Administrative"]][["Facility","Beds_Total","Hospital_Cost"]])


In [ ]:
# View
view_cols = ["Facility","Town__Facility_","Size_Category",
             "Beds_Total","Hospital_Cost","Institution","Hospital_Emissions",
             "Is_Administrative"]
master[view_cols].head(10)


## 12. Exploratory data analysis — hospitals

### 12.1 Hospitals by size category

In [ ]:
size_summary = master.groupby("Size_Category").agg(
    n_hospitals=("Facility","count"),
    total_beds=("Beds_Total","sum"),
    avg_cost_kCHF=("Hospital_Cost", lambda x: x.mean()/1e3),
    total_cost_MCHF=("Hospital_Cost", lambda x: x.sum()/1e6),
).sort_values("total_beds", ascending=False)
print(size_summary)


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.bar(size_summary.index, size_summary["n_hospitals"], color="steelblue")
plt.ylabel("Number of hospitals")
plt.title("Number of hospitals by size category — Canton of Vaud")
plt.tight_layout()
plt.show()


**Interpretation.** The Canton has many *Small* facilities (clinics, rehabilitation centres) but only a handful of large ones. Most of the **bed capacity** and most of the **running cost** are concentrated in the one *Major Center* (CHUV) and the two *Big* hospitals (Morges, Rennaz). A cost-minimizing model will be very tempted to close large hospitals — but they are exactly the ones that have enough capacity to serve dense areas, so the capacity constraint will pull back in the opposite direction.

### 12.2 Top hospitals by capacity

In [ ]:
top15 = master.sort_values("Beds_Total", ascending=False).head(15)
plt.figure(figsize=(10, 6))
plt.barh(top15["Facility"], top15["Beds_Total"], color="steelblue")
plt.gca().invert_yaxis()
plt.xlabel("Beds (per facility)")
plt.title("Top 15 hospitals by bed capacity")
plt.tight_layout()
plt.show()


### 12.3 Cost distribution

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.hist(master["Hospital_Cost"]/1e3, bins=30, color="orange", edgecolor="black")
plt.xlabel("Hospital running cost (kCHF / year, total per hospital)")
plt.ylabel("Number of hospitals")
plt.title("Distribution of hospital running cost")
plt.tight_layout()
plt.show()

print(f"Total running cost (all 50 hospitals) : {master['Hospital_Cost'].sum()/1e6:,.2f} MCHF")
print(f"Total running cost excluding CHUV     : {master[master['Size_Category']!='Major Center']['Hospital_Cost'].sum()/1e6:,.2f} MCHF")


**Interpretation.** CHUV alone (5.17 MCHF) accounts for about 42 % of the total running-cost proxy of the whole network (12.17 MCHF). Closing any small clinic only saves a small amount of money, while CHUV is irreplaceable for Lausanne. We expect the cost model to focus its savings on the *Moderate* facilities and to be forced to keep the *Big* ones.

### 12.4 Emissions distribution

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.hist(master["Hospital_Emissions"], bins=30, color="seagreen", edgecolor="black")
plt.xlabel("Allocated operational emissions (t CO2 / year)")
plt.ylabel("Number of hospitals")
plt.title("Distribution of allocated operational emissions per facility")
plt.tight_layout()
plt.show()

print(f"Total allocated emissions  : {master['Hospital_Emissions'].sum():,.0f} t CO2 / year")
print(f"CHUV (1 facility) accounts : {master.loc[master['Facility']=='Centre Hospitalier Universitaire Vaudois','Hospital_Emissions'].iloc[0]:,.0f} t CO2 / year")


**Interpretation.** The emission picture mirrors the cost picture: CHUV's share of the CHUV institution's emissions (8 143 t CO₂ × 1175 / 1300 beds ≈ 7 360 t CO₂) is the largest single contributor. The same conclusion holds: closing small facilities barely reduces operational emissions; any meaningful reduction would require structural changes at CHUV.

## 13. Exploratory data analysis — population and demand

In [ ]:
pop_by_district = pop.groupby("District", as_index=False).agg(
    Population=("Population","sum"),
    Nb_Communes=("Commune","count"),
).sort_values("Population", ascending=False)
pop_by_district["Demand_Beds"] = pop_by_district["Population"] * HOSPITALIZATION_RATE * AVG_LENGTH_OF_STAY / DAYS_PER_YEAR
print(pop_by_district)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(pop_by_district["District"], pop_by_district["Population"], color="darkorange")
axes[0].set_ylabel("Population 2024"); axes[0].set_title("Population by district")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(pop_by_district["District"], pop_by_district["Nb_Communes"], color="steelblue")
axes[1].set_ylabel("Number of communes"); axes[1].set_title("Number of communes by district")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()


**Interpretation.** Lausanne district concentrates the bulk of the population in only a few communes. Jura-Nord vaudois has the largest *number* of (mostly small, rural) communes. The optimization will therefore tend to keep at least one large hospital around Lausanne, and at least one covering hospital in Jura-Nord vaudois.

In [ ]:
# Total demand vs total supply
total_demand = dist["Demand_Beds"].sum()
total_supply = master["Beds_Total"].sum()
print(f"Total simultaneous bed demand (commune-level sum) : {total_demand:,.1f}")
print(f"Total per-facility bed supply (Beds_Total sum)    : {total_supply:,.0f}")
print(f"System-wide utilization                          : {total_demand/total_supply:.1%}")


**Interpretation.** Total demand ≈ 1942 beds against 3619 beds of capacity ⇒ system-wide utilization ≈ 54 %. There is comfortable global slack, but as we will see in notebook 2 the slack is **not uniform**: in some isolated pockets (Payerne area, Pays-d'Enhaut), the only nearby hospital has limited capacity, which constrains the smallest feasible distance threshold.

## 14. Current system baseline — all hospitals open

This is **not** an optimization. We just assume every hospital is open, and we assign each commune to its closest hospital. The resulting KPIs are the reference against which every optimized scenario in notebook 2 will be compared.

### Transport-emission formula

We model the population travelling to its closest hospital with a transport emission factor:

$$ \text{Transport\_Emissions} = e_{km} \sum_i P_i \cdot d_i^{\text{closest}} $$

with $e_{km} = 0.12$ kg CO₂ / passenger-km (assumption — change `EMISSION_FACTOR_KM` to test sensitivity).


In [ ]:
# For each commune, find the closest hospital among the 50
dM = dist[hospital_cols_dm].values
closest_idx  = dM.argmin(axis=1)
closest_dist = dM.min(axis=1)
closest_name = [hospital_cols_dm[k] for k in closest_idx]

dist["Closest_Distance_km"] = closest_dist
dist["Closest_Hospital"]    = closest_name

# KPIs
pop_total      = dist["Population"].sum()
avg_dist       = closest_dist.mean()
wavg_dist      = (closest_dist * dist["Population"]).sum() / pop_total
max_dist       = closest_dist.max()
transport_emis = EMISSION_FACTOR_KM * (closest_dist * dist["Population"]).sum() / 1000.0   # tonnes
hospital_emis  = master["Hospital_Emissions"].sum()
total_cost     = master["Hospital_Cost"].sum()
total_beds     = master["Beds_Total"].sum()
total_demand   = dist["Demand_Beds"].sum()

baseline_kpis = pd.DataFrame({
    "KPI": [
        "Open hospitals",
        "Total running cost (MCHF)",
        "Total beds (per facility)",
        "Total demand in beds (simultaneous)",
        "Average distance (km)",
        "Pop-weighted avg distance (km)",
        "Maximum distance (km)",
        "Transport emissions (t CO2)",
        "Hospital operational emissions (t CO2)",
        "Total emissions (t CO2)",
        "Capacity utilization",
    ],
    "Value": [
        len(master),
        round(total_cost/1e6, 2),
        int(total_beds),
        round(total_demand, 1),
        round(avg_dist, 2),
        round(wavg_dist, 2),
        round(max_dist, 2),
        round(transport_emis, 1),
        round(hospital_emis, 1),
        round(transport_emis + hospital_emis, 1),
        f"{total_demand/total_beds:.1%}",
    ],
})
baseline_kpis


In [ ]:
# Closest distance by district (district = reporting only, not a decision unit)
dist_by_district = dist.groupby("District", as_index=False).agg(
    Avg_dist_km=("Closest_Distance_km","mean"),
    Max_dist_km=("Closest_Distance_km","max"),
    Population=("Population","sum"),
).sort_values("Avg_dist_km", ascending=False)
dist_by_district["Avg_dist_km"] = dist_by_district["Avg_dist_km"].round(2)
dist_by_district["Max_dist_km"] = dist_by_district["Max_dist_km"].round(2)
dist_by_district


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.hist(dist["Closest_Distance_km"], bins=30, color="seagreen", edgecolor="black")
plt.axvline(D_DEFAULT_KM, color="red", linestyle="--", label=f"D = {D_DEFAULT_KM} km")
plt.xlabel("Distance to closest hospital (km)")
plt.ylabel("Number of communes")
plt.title("Distribution of closest-hospital distances (all hospitals open)")
plt.legend(); plt.tight_layout(); plt.show()


**Interpretation.** With all 50 hospitals open, the population-weighted average distance is only ~2.8 km — most Vaudois live very close to a facility. The most exposed districts are **Aigle**, **Pays-d'Enhaut** and **Jura-Nord vaudois** (10 – 16 km average closest distance). Those are the areas to watch in the optimization: closing the wrong hospital there can immediately push access above acceptable thresholds.

## 15. Export the clean datasets for the optimization notebook

We export six CSVs that notebook 2 will reload. The hospital list is in the **same row order** as the columns of the distance matrix, so notebook 2 can index by position without any merge.


In [ ]:
# 1) clean_hospitals.csv -- master hospital table
clean_hospitals = master[["Facility","Town__Facility_","longitude","latitude",
                          "Size_Category","Beds_Total",
                          "Hospital_Cost","Institution","Hospital_Emissions",
                          "Is_Administrative"]].copy()
clean_hospitals.to_csv("clean_hospitals.csv", index=False)

# 2) clean_communes.csv -- commune + district + population by year + demand + closest
clean_communes = dist[["Commune","District","Population","Demand_Beds",
                       "Closest_Distance_km","Closest_Hospital"] + list(YEAR_COLS.values())].copy()
clean_communes.to_csv("clean_communes.csv", index=False)

# 3) clean_distance_matrix.csv -- commune x hospital, in km
clean_distance = dist[["Commune"] + hospital_cols_dm].copy()
clean_distance.to_csv("clean_distance_matrix.csv", index=False)

# 4) clean_costs.csv
clean_costs = master[["Facility","Size_Category","Beds_Total","Hospital_Cost"]].copy()
clean_costs.to_csv("clean_costs.csv", index=False)

# 5) clean_emissions.csv
clean_emissions = master[["Facility","Institution","Hospital_Emissions"]].copy()
clean_emissions.to_csv("clean_emissions.csv", index=False)

# 6) clean_baseline_current_system.csv -- KPIs of "all hospitals open"
baseline_kpis.to_csv("clean_baseline_current_system.csv", index=False)

print("Files exported in the Colab working directory:")
print("  - clean_hospitals.csv")
print("  - clean_communes.csv")
print("  - clean_distance_matrix.csv")
print("  - clean_costs.csv")
print("  - clean_emissions.csv")
print("  - clean_baseline_current_system.csv")


### How notebook 2 uses these files

If you run both notebooks in the same Colab session, the files are already on disk and notebook 2 picks them up directly. If you run them in different sessions, notebook 2 falls back to re-running a minimal version of the cleaning from the raw GitHub files — so it always works.
